# 📋 Data Validation — Pydantic Schemas for 35 Clinical Features

> **Module 2 of 8** — The Cervical Cancer Prediction Service

---

## 🎯 What This Module Does

This notebook demonstrates how the API **validates incoming patient data** using **Pydantic v2** schemas. Every one of the **35 clinical features** is defined with a type, a valid range, and an alias that maps the JSON key to the original column name used during model training.

## 🛡️ Why Data Validation Matters

| Risk Without Validation | Solution With Pydantic |
|------------------------|------------------------|
| Negative age accepted | `ge=10` rejects ages < 10 |
| Binary fields like `Smokes` = 5 | Validator enforces 0 or 1 |
| Unknown/misspelled fields silently ignored | `extra="forbid"` rejects them |
| Wrong types (string instead of int) | Type coercion + validation |
| Snake_case vs original column names | `alias` maps both |

---

## 📊 The 35 Clinical Features

| # | Field (snake_case) | Alias (original) | Type | Range |
|---|-------------------|------------------|------|-------|
| 1 | `age` | `Age` | int | 10–100 |
| 2 | `number_of_sexual_partners` | `Number of sexual partners` | int | 0–100 |
| 3 | `first_sexual_intercourse` | `First sexual intercourse` | int | 10–80 |
| 4 | `num_of_pregnancies` | `Num of pregnancies` | int | 0–20 |
| 5 | `smokes` | `Smokes` | int | 0–1 |
| 6 | `smokes_years` | `Smokes (years)` | int | 0–50 |
| 7 | `smokes_packs_per_year` | `Smokes (packs/year)` | float | 0–100 |
| 8 | `hormonal_contraceptives` | `Hormonal Contraceptives` | int | 0–1 |
| 9 | `hormonal_contraceptives_years` | `Hormonal Contraceptives (years)` | float | 0–30 |
| 10 | `iud` | `IUD` | int | 0–1 |
| 11 | `iud_years` | `IUD (years)` | float | 0–30 |
| 12 | `stds` | `STDs` | int | 0–1 |
| 13 | `stds_number` | `STDs (number)` | int | 0–10 |
| 14–25 | `stds_*` (12 STD flags) | `STDs:*` | int | 0–1 |
| 26 | `stds_number_of_diagnosis` | `STDs: Number of diagnosis` | int | 0–10 |
| 27 | `stds_time_since_first_diagnosis` | `STDs: Time since first diagnosis` | float | 0–50 |
| 28 | `stds_time_since_last_diagnosis` | `STDs: Time since last diagnosis` | float | 0–50 |
| 29–32 | `dx_*` (4 diagnosis flags) | `Dx:*` | int | 0–1 |
| 33–35 | `hinselmann`, `schiller`, `citology` | (same) | int | 0–1 |

---

## 🧠 Skills Demonstrated

- ✅ **Pydantic v2** — modern, type-safe data validation
- ✅ **API Contract Design** — precise request/response schemas
- ✅ **Field Validators** — custom business-rule validation
- ✅ **Alias Mapping** — snake_case ↔ original column names

In [ ]:
# ── Add the api directory to sys.path so 'app' module can be found ──
import sys
from pathlib import Path

_candidates = [
    Path.cwd() / "cervical-cancer-mlops" / "api",
    Path.cwd().parent / "api",
    Path.cwd() / "api",
]
api_dir = next((p for p in _candidates if (p / "app").exists()), None)
if api_dir is None:
    raise RuntimeError(
        "Could not locate the 'api/app' package. "
        "Expected it under one of: " + ", ".join(map(str, _candidates))
    )
if str(api_dir) not in sys.path:
    sys.path.insert(0, str(api_dir))
print(f"✅ Added {api_dir} to sys.path")


In [ ]:
# ── Import the Pydantic schemas ──
from app.models import PredictionInput, PredictionOutput, HealthResponse, ReportOutput, ErrorResponse

print("✅ Imported all Pydantic schemas:")
print(f"   - PredictionInput  ({len(PredictionInput.model_fields)} fields)")
print(f"   - PredictionOutput")
print(f"   - HealthResponse")
print(f"   - ReportOutput")
print(f"   - ErrorResponse")

In [ ]:
# ── Demonstrate VALID input passing validation ──
valid_patient = {
    "age": 25,
    "number_of_sexual_partners": 1,
    "first_sexual_intercourse": 22,
    "num_of_pregnancies": 0,
    "smokes": 0,
    "smokes_years": 0,
    "smokes_packs_per_year": 0.0,
    "hormonal_contraceptives": 0,
    "hormonal_contraceptives_years": 0.0,
    "iud": 0,
    "iud_years": 0.0,
    "stds": 0,
    "stds_number": 0,
    "stds_condylomatosis": 0,
    "stds_cervical_condylomatosis": 0,
    "stds_vaginal_condylomatosis": 0,
    "stds_vulvo_perineal_condylomatosis": 0,
    "stds_syphilis": 0,
    "stds_pelvic_inflammatory_disease": 0,
    "stds_genital_herpes": 0,
    "stds_molluscum_contagiosum": 0,
    "stds_aids": 0,
    "stds_hiv": 0,
    "stds_hepatitis_b": 0,
    "stds_hpv": 0,
    "stds_number_of_diagnosis": 0,
    "stds_time_since_first_diagnosis": 0.0,
    "stds_time_since_last_diagnosis": 0.0,
    "dx_cancer": 0,
    "dx_cin": 0,
    "dx_hpv": 0,
    "dx": 0,
    "hinselmann": 0,
    "schiller": 0,
    "citology": 0
}

try:
    validated = PredictionInput(**valid_patient)
    print("✅ VALID input accepted!")
    print(f"   Age: {validated.age}, Partners: {validated.number_of_sexual_partners}")
    print(f"   Dumped with aliases: {list(validated.model_dump(by_alias=True).keys())[:3]}...")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

In [ ]:
# ── Demonstrate INVALID input being rejected ──
from pydantic import ValidationError

# Test 1: Negative age
bad_age = dict(valid_patient, age=-5)
try:
    PredictionInput(**bad_age)
    print("❌ Should have failed!")
except ValidationError as e:
    print("✅ Rejected negative age:")
    print(f"   {e.errors()[0]['msg']}")

# Test 2: Binary field = 5
bad_binary = dict(valid_patient, smokes=5)
try:
    PredictionInput(**bad_binary)
    print("❌ Should have failed!")
except ValidationError as e:
    print("✅ Rejected smokes=5 (must be 0 or 1):")
    print(f"   {e.errors()[0]['msg']}")

# Test 3: Unknown field
bad_extra = dict(valid_patient, unknown_field=123)
try:
    PredictionInput(**bad_extra)
    print("❌ Should have failed!")
except ValidationError as e:
    print("✅ Rejected unknown field:")
    print(f"   {e.errors()[0]['msg']}")

---

## ✅ Summary

The Pydantic schemas provide a **strict, self-documenting API contract**. Invalid data is rejected with clear error messages before it ever reaches the model — protecting the ML pipeline from garbage input.

**Next:** [Module 3 — Input Preprocessing](api_03_preprocessor.ipynb)